<a href="https://colab.research.google.com/github/r3mo-usd/AAI-500-02-final-project-Sahana-Javier/blob/feature%2Fdata-cleaning-eda/src/javier_da.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Config

**Introduction**

When a customer cancels their service, it is called **customer churn**. Customers often do not leave after a single bad experience. They usually cancel after repeated disappointments, such as billing problems or poor service. Once they leave, the company loses that customer's ongoing revenue and has to spend money to replace them. Attracting a new customer is more expensive than keeping an existing one.

This project uses the Telco Customer Churn dataset, a publicly available sample dataset originally released by IBM. It contains information on 7,043 customers, including demographics, account details (contract type, tenure, payment method), services subscribed to, and monthly and total charges. The target variable is Churn (Yes/No), and 26.5% of customers in the dataset have churned.

We want to identify the reasons behind customer churn and understand which factors influence it, so the company can improve customer retention. If we can find the warning signs early, the company can act before the customer decides to leave. To do this, we first clean and explore the data to see which characteristics are related to churn. We then build a logistic regression model to test and measure those relationships.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


### **Load the data**

In [2]:
# load the dataset from the shared GitHub repo (main branch)
url = 'https://raw.githubusercontent.com/r3mo-usd/AAI-500-02-final-project-Sahana-Javier/main/data/WA_Fn-UseC_-Telco-Customer-Churn.csv'
telco = pd.read_csv(url)


In [3]:
# first 5 observations
print(telco.head(5))

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

## **Check variable types**

In [4]:
# dimensions: 7043 rows, 21 columns
print(telco.shape)

# non-missing counts and data type for each variable
# TotalCharges shows up as object (text) instead of a number
print(telco.info())

(7043, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null

TotalCharges is stored as text, so it must be converted to numeric before analysis.

### **Convert TotalCharges**

In [5]:
# TotalCharges is stored as text; convert to numeric
telco['TotalCharges'] = pd.to_numeric(telco['TotalCharges'], errors='coerce')

# count the missing values created by the conversion
telco['TotalCharges'].isna().sum()

np.int64(11)

**Inspect the 11 blanks**

In [6]:
# check which customers have missing TotalCharges
telco.loc[telco['TotalCharges'].isna()][['tenure', 'MonthlyCharges', 'TotalCharges']]


,tenure,MonthlyCharges,TotalCharges
488,0,52.55,NaN
753,0,20.25,NaN
936,0,80.85,NaN
1082,0,25.75,NaN
1340,0,56.05,NaN
3331,0,19.85,NaN
3826,0,25.35,NaN
4380,0,20.00,NaN
5218,0,19.70,NaN
6670,0,73.35,NaN


**Fill the blanks**

In [7]:
# fill missing TotalCharges with 0 (new customers have not been charged yet)
telco['TotalCharges'] = telco['TotalCharges'].fillna(0)

# check that no missing values are left (should show 0)
telco['TotalCharges'].isna().sum()

np.int64(0)

We set these values to 0 instead of dropping the rows, so all 7,043 customers stay in the sample.

**Drop customerID**

In [8]:
# drop customerID because it is just an ID number for each customer
telco = telco.drop(['customerID'], axis=1)
telco.shape

(7043, 20)

In [9]:
#change SeniorCitizen from 1/0 to Yes/No so it matches the other columnsprint
(telco["SeniorCitizen"].value_counts())
telco["SeniorCitizen"] = telco["SeniorCitizen"].replace({1: "Yes", 0: "No"})

# check the new values (show only No and Yes)
print(telco["SeniorCitizen"].value_counts())

SeniorCitizen
No     5901
Yes    1142
Name: count, dtype: int64


In [10]:
# check if "No internet service" only shows up for customers with no internet
pd.crosstab(telco["InternetService"], telco["TechSupport"])

TechSupport,No,No internet service,Yes
InternetService,,,
DSL,1243,0,1178
Fiber optic,2230,0,866
No,0,1526,0


The table shows that 'No internet service' appears only for the 1,526 customers whose InternetService is 'No'. Since InternetService already records this, we collapsed it into 'No' to avoid repeating the same information.

### Simplifying the 'No Service'

In [14]:
# these six columns have "No internet service" as a third option
service_columns = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                    "TechSupport", "StreamingTV", "StreamingMovies"]

for col in service_columns:
    telco[col] = telco[col].replace({"No internet service": "No"})

# do the same for MultipleLines, where "No phone service" becomes "No"
telco["MultipleLines"] = telco["MultipleLines"].replace({"No phone service": "No"})

# check that only Yes and No are left in each column
for col in service_columns + ["MultipleLines"]:
    print(telco[col].value_counts())
    print()


OnlineSecurity
No     5024
Yes    2019
Name: count, dtype: int64

OnlineBackup
No     4614
Yes    2429
Name: count, dtype: int64

DeviceProtection
No     4621
Yes    2422
Name: count, dtype: int64

TechSupport
No     4999
Yes    2044
Name: count, dtype: int64

StreamingTV
No     4336
Yes    2707
Name: count, dtype: int64

StreamingMovies
No     4311
Yes    2732
Name: count, dtype: int64

MultipleLines
No     4072
Yes    2971
Name: count, dtype: int64



OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, and StreamingMovies each had a third value called 'No internet service.' This only showed up for customers who don't have internet at all, so it's really the same information as the InternetService column. MultipleLines had the same thing with 'No phone service.' Since we already have that information in another column, we changed these to just 'No' so we're not repeating the same thing over and over.

In [12]:
# final check: 7043 rows, 20 columns, no missing values, TotalCharges is float64
telco.info()
telco.shape

# save the cleaned dataset for modeling
telco.to_csv("telco_cleaned.csv", index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   object 
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


In [15]:
from google.colab import files
files.download("telco_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# EDA

## General info

In [13]:
print(telco.describe())

            tenure  MonthlyCharges  TotalCharges
count  7043.000000     7043.000000   7043.000000
mean     32.371149       64.761692   2279.734304
std      24.559481       30.090047   2266.794470
min       0.000000       18.250000      0.000000
25%       9.000000       35.500000    398.550000
50%      29.000000       70.350000   1394.550000
75%      55.000000       89.850000   3786.600000
max      72.000000      118.750000   8684.800000
